In [1]:
# Load CheXpert+ metadata, labels, and image paths.
# Then select one frontal image per study, so each final row is one study.
# Create a probe train/test split and save a CSV.

from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 100)

/home/tdnguyen/miniforge3/envs/cxr-vlm-interp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set paths, labels, and split sizes.

CXR_DIR = Path("/opt/gpudata/cxr")
CHEXPERTPLUS_DIR = CXR_DIR / "chexpertplus"
LABEL_DIR = CXR_DIR / "derived"

IMAGE_ROOT = CHEXPERTPLUS_DIR / "PNG"
SPLIT_CSV = CHEXPERTPLUS_DIR / "split.csv"
METADATA_CSV = CHEXPERTPLUS_DIR / "metadata.csv"
LABEL_CSV = LABEL_DIR / "chexpertplus-findings-labels-chexbert.csv"

TARGET_LABELS = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Pleural Effusion",
]

TRAIN_STUDIES_N = 20_000
TEST_STUDIES_N = 5_000
RANDOM_SEED = 42

OUT_PATH = Path("artifacts/processed_data/chexpertplus_frontal_5labels.csv")

for path in [IMAGE_ROOT, SPLIT_CSV, METADATA_CSV, LABEL_CSV]:
    print(f"{path}: {path.exists()}")

/opt/gpudata/cxr/chexpertplus/PNG: True
/opt/gpudata/cxr/chexpertplus/split.csv: True
/opt/gpudata/cxr/chexpertplus/metadata.csv: True
/opt/gpudata/cxr/derived/chexpertplus-findings-labels-chexbert.csv: True


In [3]:
# Load the split table, metadata table, and CheXpert labels.

split_df = pd.read_csv(SPLIT_CSV)
meta_df = pd.read_csv(METADATA_CSV)
label_df = pd.read_csv(LABEL_CSV)

print("split_df", split_df.shape)
display(split_df.head())

print("meta_df", meta_df.shape)
display(meta_df.head())

print("label_df", label_df.shape)
display(label_df[["study_id"] + TARGET_LABELS].head())

split_df (223461, 4)


,subject_id,study_id,dicom_id,split
0,patient00001,patient00001_study1,patient00001_study1_view1_frontal,train
1,patient00002,patient00002_study1,patient00002_study1_view1_frontal,train
2,patient00002,patient00002_study1,patient00002_study1_view2_lateral,train
3,patient00002,patient00002_study2,patient00002_study2_view1_frontal,train
4,patient00003,patient00003_study1,patient00003_study1_view1_frontal,train


meta_df (223461, 5)


,subject_id,study_id,dicom_id,frontal_lateral,ViewPosition
0,patient00001,patient00001_study1,patient00001_study1_view1_frontal,Frontal,AP
1,patient00002,patient00002_study1,patient00002_study1_view1_frontal,Frontal,AP
2,patient00002,patient00002_study1,patient00002_study1_view2_lateral,Lateral,Lateral
3,patient00002,patient00002_study2,patient00002_study2_view1_frontal,Frontal,AP
4,patient00003,patient00003_study1,patient00003_study1_view1_frontal,Frontal,AP


label_df (46759, 16)


,study_id,Atelectasis,Cardiomegaly,Consolidation,Edema,Pleural Effusion
0,patient00001_study1,1,0,0,0,0
1,patient00002_study1,0,0,1,0,0
2,patient00003_study1,0,0,0,1,0
3,patient00005_study2,1,0,1,0,1
4,patient00008_study1,1,0,1,0,1


In [4]:
# Scan image files and create IDs that match the metadata tables.
# CheXpert+ image paths are expected to end with patient_id / raw_study_id / raw_dicom_id.png.
# The normalized IDs are study_id = patient_id_raw_study_id and dicom_id = study_id_raw_dicom_id.

rows = []
image_paths = sorted(IMAGE_ROOT.rglob("*.png"))

for path in tqdm(image_paths, desc="Scanning images"):
    patient_id, raw_study_id, raw_dicom_id = path.with_suffix("").parts[-3:]
    study_id = f"{patient_id}_{raw_study_id}"
    dicom_id = f"{study_id}_{raw_dicom_id}"
    rows.append(
        {
            "subject_id": patient_id,
            "study_id": study_id,
            "dicom_id": dicom_id,
            "image_path": str(path),
        }
    )

image_df = pd.DataFrame(rows)

# patient32368 has an image-loading problem.
image_df = image_df[image_df["subject_id"] != "patient32368"].reset_index(drop=True)

print("image_df", image_df.shape)
display(image_df.head())

Scanning images: 100%|██████████| 223461/223461 [00:00<00:00, 337173.23it/s]


image_df (223461, 4)


,subject_id,study_id,dicom_id,image_path
0,patient00001,patient00001_study1,patient00001_study1_view1_frontal,/opt/gpudata/cxr/chexpertplus/PNG/train/patien...
1,patient00002,patient00002_study1,patient00002_study1_view1_frontal,/opt/gpudata/cxr/chexpertplus/PNG/train/patien...
2,patient00002,patient00002_study1,patient00002_study1_view2_lateral,/opt/gpudata/cxr/chexpertplus/PNG/train/patien...
3,patient00002,patient00002_study2,patient00002_study2_view1_frontal,/opt/gpudata/cxr/chexpertplus/PNG/train/patien...
4,patient00003,patient00003_study1,patient00003_study1_view1_frontal,/opt/gpudata/cxr/chexpertplus/PNG/train/patien...


In [5]:
# Merge split, metadata, image paths, and labels.

meta_small = meta_df[["subject_id", "study_id", "dicom_id", "ViewPosition"]].copy()
meta_small["view"] = meta_small["ViewPosition"].fillna("").str.upper()

data = (
    split_df[["study_id", "split"]]
    .drop_duplicates("study_id")
    .merge(meta_small, on="study_id", how="inner")
    .merge(image_df, on=["subject_id", "study_id", "dicom_id"], how="left")
    .merge(label_df[["study_id"] + TARGET_LABELS], on="study_id", how="inner")
)

data["has_image"] = data["image_path"].notna()

print("data", data.shape)
display(data.head())

data (57806, 13)


,study_id,split,subject_id,dicom_id,ViewPosition,view,image_path,Atelectasis,Cardiomegaly,Consolidation,Edema,Pleural Effusion,has_image
0,patient00001_study1,train,patient00001,patient00001_study1_view1_frontal,AP,AP,/opt/gpudata/cxr/chexpertplus/PNG/train/patien...,1,0,0,0,0,True
1,patient00002_study1,train,patient00002,patient00002_study1_view1_frontal,AP,AP,/opt/gpudata/cxr/chexpertplus/PNG/train/patien...,0,0,1,0,0,True
2,patient00002_study1,train,patient00002,patient00002_study1_view2_lateral,Lateral,LATERAL,/opt/gpudata/cxr/chexpertplus/PNG/train/patien...,0,0,1,0,0,True
3,patient00003_study1,train,patient00003,patient00003_study1_view1_frontal,AP,AP,/opt/gpudata/cxr/chexpertplus/PNG/train/patien...,0,0,0,1,0,True
4,patient00005_study2,train,patient00005,patient00005_study2_view1_frontal,AP,AP,/opt/gpudata/cxr/chexpertplus/PNG/train/patien...,1,0,1,0,1,True


In [6]:
# Print basic statistics before filtering.

print("Rows:", len(data))
print("Studies:", data["study_id"].nunique())
print("Patients:", data["subject_id"].nunique())

print("View distribution")
display(data["view"].value_counts(dropna=False).to_frame("rows"))

print("Rows with image paths:", int(data["has_image"].sum()))

Rows: 57806
Studies: 46759
Patients: 26695
View distribution


,rows
view,
AP,39384
LATERAL,9573
PA,8842
LL,6
RL,1


Rows with image paths: 57806


In [7]:
# Keep PA/AP images only.
# If a study has multiple frontal images, prefer PA, then AP.
# After this step, each row is one study.

frontal = data[data["view"].isin(["PA", "AP"]) & data["has_image"]].copy()
frontal["view_rank"] = frontal["view"].map({"PA": 0, "AP": 1})

manifest = (
    frontal.sort_values(["study_id", "view_rank", "dicom_id"])
    .drop_duplicates("study_id", keep="first")
    .sort_values(["split", "subject_id", "study_id"])
    .reset_index(drop=True)
)

print("manifest", manifest.shape)
display(manifest.head())

print("Rows:", len(manifest))
print("Studies:", manifest["study_id"].nunique())
print("Patients:", manifest["subject_id"].nunique())

print("View distribution after filtering")
display(manifest["view"].value_counts(dropna=False).to_frame("rows"))

print("Each row is one study:", len(manifest) == manifest["study_id"].nunique())

manifest (46746, 14)


,study_id,split,subject_id,dicom_id,ViewPosition,view,image_path,Atelectasis,Cardiomegaly,Consolidation,Edema,Pleural Effusion,has_image,view_rank
0,patient64544_study1,test,patient64544,patient64544_study1_view1_frontal,AP,AP,/opt/gpudata/cxr/chexpertplus/PNG/valid/patien...,0,0,0,0,0,True,1
1,patient64545_study1,test,patient64545,patient64545_study1_view1_frontal,AP,AP,/opt/gpudata/cxr/chexpertplus/PNG/valid/patien...,1,0,0,1,1,True,1
2,patient64548_study1,test,patient64548,patient64548_study1_view1_frontal,AP,AP,/opt/gpudata/cxr/chexpertplus/PNG/valid/patien...,0,0,0,0,0,True,1
3,patient64555_study1,test,patient64555,patient64555_study1_view1_frontal,AP,AP,/opt/gpudata/cxr/chexpertplus/PNG/valid/patien...,0,0,0,0,0,True,1
4,patient64564_study1,test,patient64564,patient64564_study1_view1_frontal,AP,AP,/opt/gpudata/cxr/chexpertplus/PNG/valid/patien...,0,0,0,0,0,True,1


Rows: 46746
Studies: 46746
Patients: 26687
View distribution after filtering


,rows
view,
AP,38178
PA,8568


Each row is one study: True


In [8]:
# Create a probe split where each row is one study.
# MedGemma was not trained on CheXpert+, so we do not need to follow the original train/validate/test split.
# We split the data ourselves: 20,000 train studies and 5,000 test studies.
# We still assign split membership at the patient level, so the same patient cannot appear in both train and test.

patient_counts = (
    manifest.groupby("subject_id")
    .size()
    .rename("n_rows")
    .reset_index()
    .sample(frac=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

train_patients = []
test_patients = []
train_rows = 0
test_rows = 0

for row in patient_counts.itertuples(index=False):
    if train_rows + row.n_rows <= TRAIN_STUDIES_N:
        train_patients.append(row.subject_id)
        train_rows += row.n_rows
    elif test_rows + row.n_rows <= TEST_STUDIES_N:
        test_patients.append(row.subject_id)
        test_rows += row.n_rows
    if train_rows == TRAIN_STUDIES_N and test_rows == TEST_STUDIES_N:
        break

manifest["probe_split"] = "unused"
manifest.loc[manifest["subject_id"].isin(train_patients), "probe_split"] = "train"
manifest.loc[manifest["subject_id"].isin(test_patients), "probe_split"] = "test"

probe_data = manifest[manifest["probe_split"].isin(["train", "test"])].copy()

print("probe_data", probe_data.shape)
display(
    probe_data.groupby("probe_split")
    .agg(rows=("study_id", "size"), studies=("study_id", "nunique"), patients=("subject_id", "nunique"))
)

patient_probe_split_counts = probe_data.groupby("subject_id")["probe_split"].nunique()
print("Patients in both train and test:", int((patient_probe_split_counts > 1).sum()))

probe_data (25000, 15)


,rows,studies,patients
probe_split,,,
test,5000,5000,2909
train,20000,20000,11453


Patients in both train and test: 0


In [9]:
# Print label statistics for the five target findings.
# The primary probing rule will be positive = label == 1, and everything else is negative.

label_rows = []

for split, split_data in probe_data.groupby("probe_split"):
    for label in TARGET_LABELS:
        s = split_data[label]
        label_rows.append(
            {
                "probe_split": split,
                "label": label,
                "n": len(s),
                "positive_1": int((s == 1).sum()),
                "negative_0": int((s == 0).sum()),
                "uncertain_-1": int((s == -1).sum()),
                "missing": int(s.isna().sum()),
                "primary_positive_rate": (s == 1).mean(),
            }
        )

label_stats = pd.DataFrame(label_rows)
display(label_stats)

,probe_split,label,n,positive_1,negative_0,uncertain_-1,missing,primary_positive_rate
0,test,Atelectasis,5000,1522,3478,0,0,0.30440
1,test,Cardiomegaly,5000,1098,3902,0,0,0.21960
2,test,Consolidation,5000,876,4124,0,0,0.17520
3,test,Edema,5000,1377,3623,0,0,0.27540
4,test,Pleural Effusion,5000,2286,2714,0,0,0.45720
5,train,Atelectasis,20000,6186,13814,0,0,0.30930
6,train,Cardiomegaly,20000,4440,15560,0,0,0.22200
7,train,Consolidation,20000,3599,16401,0,0,0.17995
8,train,Edema,20000,5815,14185,0,0,0.29075
9,train,Pleural Effusion,20000,9504,10496,0,0,0.47520


In [10]:
# Save the processed dataset.

columns_to_save = [
    "subject_id",
    "study_id",
    "dicom_id",
    "split",
    "probe_split",
    "ViewPosition",
    "view",
    "image_path",
] + TARGET_LABELS

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
probe_data[columns_to_save].to_csv(OUT_PATH, index=False)

print(f"Saved {len(probe_data):,} rows to {OUT_PATH}")

Saved 25,000 rows to artifacts/processed_data/chexpertplus_frontal_5labels.csv
